(course-core-extracting-and-removing-elements)=

# Module 18: Extracting and Removing Elements

**Welcome back, Apprentice Master.** In [Module 17: Merging and Growing Systems](../Common_Core/17_Merging_and_Growing_Systems.ipynb), you learned how to assemble complex molecular systems using `msm.merge()` and `msm.add()`. Now we explore the inverse operations: **Extracting and Removing Elements**.

Large simulation boxes often contain tens of thousands of water molecules, counterions, or bulk solvent. Analyzing a binding pocket or isolated protein domain requires cutting away non-essential components without corrupting atomic indices or structural topology. MolSysMT provides dedicated extraction and removal engines (**`msm.extract()`** and **`msm.remove()`**) for precise sub-system manipulation.

(course-core-extracting-and-removing-elements-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Extract isolated sub-systems into new objects using `msm.extract()`.
- Purge unwanted molecules (such as water or ions) using `msm.remove()`.
- Verify structural and topological consistency after element extraction and removal.
```

### 1. Extracting Sub-Systems

Let's begin by importing MolSysMT and loading our T4 Lysozyme demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme as native MolSys
lysozyme = msm.convert(systems['T4 lysozyme L99A']['181l.bcif.gz'], to_form='molsysmt.MolSys')

`msm.extract()` creates a new independent molecular system containing only the selected atoms or components, leaving the original system untouched. A partial `MolSys` with topology and chemical states but no structures can also select atom indices: absent coordinates remain absent. With topology, atom subsets follow sorted source indices. Selecting structure indices requires a declared structure axis; topology and chemistry alone do not provide one. See the [H5MSM 0.5 guide](../../user/tools/form/file/h5msm_05.md) for supported partial-file workflows:
For a ViewerJSON graphics payload, explicit atom extraction preserves your
requested order and remaps retained bond endpoints and aligned bond columns.
Atom indices must be unique; structure lists preserve order and repetitions.
Discarded structures are not copied. This is graphics extraction, not a full
native-domain archive. See
{ref}`ViewerJSON <user-foundations-native-world-classes-molsysmt-viewerjson>`.


In [2]:
# Extract only the protein component
protein_only = msm.extract(lysozyme, selection='molecule_type == "protein"')

# Inspect topology overview of extracted protein
msm.info(protein_only)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_proteins,n_structures
molsysmt.MolSys,1289,162,1,1,1,1,1,1


:::{hint}
:class: dropdown
**msm.extract()**: Creates a new independent molecular system from selected atoms, groups, or components. See API doc: {func}`molsysmt.basic.extract`.
:::

### 2. Removing Elements

If you want to keep everything in the system *except* specific unwanted elements (such as removing solvent water or counterions), use `msm.remove()`:

The default `in_place=False` returns an independent result, including an empty removal. Use `msm.remove(molsys, selection=..., in_place=True)` only for an explicit native edit; it returns `None`. Pass the complete owning MolSys rather than resize an attached Topology or Structures. Domains and named analyses are projected together; old references remain snapshots, and no interaction detector runs. See {ref}`Tutorial_Remove`.

In [3]:
# Remove water molecules and ions from the system
dry_system = msm.remove(lysozyme, selection='molecule_type == "water" or molecule_type == "ion"')

# Inspect topology overview of dry system
msm.info(dry_system)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_small_molecules,n_proteins,n_structures
molsysmt.MolSys,1303,164,3,3,3,3,2,1,1


### Applying a declared atom order

Selection and atom permutation are different operations. A full selection with
`msm.extract()` retains its existing selection semantics. When a workflow knows
a complete atom bijection, use `msm.basic.permute_atoms()` to transport all
supported molecular domains to that new atom axis. Only atom indices change;
group, component and structure axes and original string IDs remain fixed.

The experimental operation returns an independent MolSys. It rejects unsupported
extensions; original preparation evidence remains on its historical axes.
Its output remains resident even though coordinates are processed in blocks.

:::{hint}
:class: dropdown

{func}`molsysmt.basic.permute_atoms` applies a complete new-to-old order and can
return both map directions. See {ref}`Tutorial_Permute_Atoms` for its domain and
memory boundaries.
:::


In [4]:
molsys_order_example = msm.convert(msm.systems['caffeine']['caffeine.sdf'])
reverse_order = list(range(molsys_order_example.get_n_atoms() - 1, -1, -1))
permutation = msm.basic.permute_atoms(molsys_order_example, reverse_order, return_report=True)
assert permutation['report']['output_to_source_atom_indices'][:3].tolist() == [23, 22, 21]
molsys_order_restored = msm.basic.permute_atoms(
    permutation['molecular_system'], permutation['report']['source_to_output_atom_indices'])
assert molsys_order_restored.topology.atoms.atom_id.tolist() == molsys_order_example.topology.atoms.atom_id.tolist()
print('Complete atom axis restored; other index axes retained.')


Complete atom axis restored; other index axes retained.


--- 

### 🏆 Challenge 18: The Subsystem Specialist

1. Load the **T4 Lysozyme** system (`systems['T4 lysozyme L99A']['181l.bcif.gz']`).
2. Extract groups 10 through 20 into a new object called `sub_domain` using `msm.extract()`.
3. Use `msm.remove()` to delete all water molecules from `lysozyme`.
4. Verify that the atom count of `sub_domain` matches the extracted group selection.

Subsystem extraction allows you to isolate active sites and reduce computational overhead. In [Module 19: Structures and Trajectories](../Common_Core/19_Structures_and_Trajectories.ipynb), we will explore multi-structure datasets and trajectory slicing.

```{key-takeaway}
MolSysMT provides clear subsystem isolation verbs: `msm.extract()` isolates selected sub-systems into new objects, while `msm.remove()` purges unwanted components.
```

(course-core-extracting-and-removing-elements-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.extract` — Sub-system extraction engine.
- {func}`molsysmt.basic.remove` — Element removal tool.

**Related Course Modules & Guides:**
- Previous Module: [Module 17: Merging and Growing Systems](../Common_Core/17_Merging_and_Growing_Systems.ipynb)
- Next Module: [Module 19: Structures and Trajectories](../Common_Core/19_Structures_and_Trajectories.ipynb)
- User Guide: {ref}`user-foundations-index`
:::